# PE_V2X_Reliability 项目报告（方案A）

本报告系统性呈现：研究动机、三场景建模、机制与指标口径、可复现性设计，以及 Fig01–Fig07 的结果解构与讨论。Fig01–Fig07 在正文中以 PNG 预览呈现（由权威 PDF 渲染）。

**Date:** 2026-03-01

## 1 研究动机与问题定义

车联网（V2X）安全消息的核心要求并非“带宽最大”或“平均时延最小”，而是要在**高动态拓扑**、**复杂传播环境**与**共享无线信道争用**条件下，持续满足协同感知/协同决策所需的**可靠性与时效性约束**。在实际道路中，退化环境（城市遮挡、隧道）与忙时拥塞并非罕见“极端案例”，而是工程系统必须面对的常态边界：城市街谷、立交与长隧道广泛存在；交通高峰引发的信道忙占比上升也具有普遍性。因此，围绕“退化环境下 V2X 的可靠性与时效性”建立一套**可解释、可复现、可扩展**的评估框架，具有明确的工程意义与研究价值。

从业务视角出发，V2X 安全消息的失败模式至少应被区分为两类，且二者对安全业务的影响路径不同：

1) **可靠性失败（Non-delivery）**：消息未到达，表现为 PDR（Packet Delivery Ratio）降低。  
2) **时效性失败（Late delivery）**：消息虽到达但超过截止时间（deadline），对协同决策而言等价于无效，是一种“隐性失败”。

这意味着单一指标（例如平均时延或单一 PDR）不足以支撑严谨结论：平均值往往掩盖**尾部风险（tail latency）**；而在 deadline 存在时，尾部分布的右移会直接转化为 late，从而降低业务有效成功率。为避免“现象描述而无法归因”，本项目采用**机制可解释**的系统级建模思路，并在统一协议下联合报告：

- **可靠性随距离衰减**：PDR vs distance；
- **时延分布与尾部风险**：Delay CDF、p95/p99；
- **deadline-aware 有效成功**：`success_phy`（物理成功）、`late`（超时）、`success`（及时成功）。

同时，本项目将退化环境收敛为三类代表性场景（RefPlus/UrbMaskPlus/TunnelPlus），并引入 **NoCong / Cong** 双体制对照，在保持其余协议冻结的前提下，仅通过拥塞开关引入争用与碰撞效应，以回答两个更接近“系统设计决策”的问题：

- 忙时拥塞是否成为共同主导项，从而压缩环境差异、改变策略选择？  
- 以重传次数 `ret=0/1/2` 为主增强杠杆时，收益（可靠性提升）与代价（尾时延变厚、late 增长）如何在不同场景/体制下权衡？

---

## 2 项目构建内容与总体路线

本项目的工作成果可概括为一套**系统级评估流水线**：输入生成 → 事件仿真 → 聚合统计 → 图表产出，并以 **run_id 输出契约 + 命令快照**固化为可复现实验包。其目标不是“跑出某一次结果”，而是建立一个可复用骨架，使得在后续扩展（更多退化环境、RSU、策略改进、真实轨迹输入）时仍能保持同一套指标口径与证据链。

### 2.1 输入生成层：将场景与交通显式化、可审计化

为避免“关键设定隐含在代码中导致不可复现”，本项目将输入拆分并固化为可追溯文件/配置：

- **轨迹输入（mobility）**：RefPlus 几何 + IDM 跟驰 + 信号灯相位 + 路口 cross/turn 机制生成轨迹 CSV。  
- **建筑输入（UrbMask）**：以矩形楼块集合表达街区几何，生成建筑 CSV（footprint、height、分布模式等）。  
- **隧道输入（Tunnel）**：以隧道区间与退化参数集合表达洞内退化，生成 tunnel config CSV。

这一层的关键工程价值在于：评审者不需要作者的电脑目录，也不需要理解每一行代码，就能基于输入文件与命令快照复现同一仿真条件。

### 2.2 事件仿真层：以“消息事件”为基本粒度输出包级样本

仿真核心将每条安全消息的传播结果作为基本样本单位，输出包含三类信息：

- **几何/状态量**：distance、链路中点位置、场景状态（如 NLOS/inside tunnel）等；  
- **结果量**：`success_phy`、`delay_ms`、`late`、`success`；  
- **证据字段**：遮挡强度（如 `d_min`/`b`）、隧道位置（`tunnel_u`）、拥塞代理量（`n_cs`/CBR/`p_col`/`cong_delay_ms`）等。

证据字段的存在，使得后续图表不仅能展示“变差了”，还能解释“为什么变差”。

### 2.3 聚合统计层：将 raw 压缩为审阅友好的三表体系

考虑包级 raw 可能体积较大，本项目将评审核查对齐到聚合表（CSV）：

- **summary_metrics**：按距离分箱统计 PDR、p50/p95/p99、late ratio，并汇总证据字段均值；  
- **position_heterogeneity**：UrbMask 的同距异质性表（固定距离带内按 `mid_x` 分桶）；  
- **tunnel_segments**：Tunnel 的分段统计表（固定距离带内按 `tunnel_u` 分桶）。

三表体系并非“为了多产出文件”，而是严格对应后续机制解释：Fig05 需要异质性表支撑；Fig06 需要分段表支撑；否则结论不可审计。

### 2.4 图表产出层：区分“定稿图”与“透明化旁证图”

输出图表分两类：

- **最终定稿图（Fig01–Fig07）**：口径统一、信息密度高，用于提交/答辩引用；  
- **旁证图（deliverable 的 F1/F2/F3）**：覆盖全量组合（场景×ret×seeds），用于透明化与兴趣下钻，不替代定稿口径。

---

## 3 项目意义、价值与贡献点

本节采用“价值—贡献—可核查性”的论文式表达，强调与本项目实际工作一致的意义，不做夸张表述。

### 3.1 价值：为什么这套工作重要且必要

**(1) 评价口径贴近安全业务：把“迟到”显式纳入成功定义**  
真实系统中，“收到但太晚”对协同决策等价于失败。项目显式区分 `success_phy` 与 `success`，并用 `late` 描述时效性失败，使得结论能够直接服务于安全业务约束，而不是停留在网络指标层面。

**(2) 面向退化环境的关键现象被机制化：异质性与分段效应可被量化**  
城市街谷的主要难点不在于“平均更差”，而在于同距异质性；隧道的主要难点不在于“整体更差”，而在于分段退化与尾部变厚。项目通过 `position_heterogeneity` 与 `tunnel_segments` 将这两类现象从“直觉描述”提升为“可核查统计结果”。

**(3) 拥塞与增强策略的权衡可被解释：避免单指标误导**  
重传 ret 的收益往往伴随 tail 代价；在拥塞下，PHY 成功率提升可能被 late 增长抵消。项目通过机制拆解（Fig03）与拥塞代理证据（Fig04）把“权衡”从口头讨论落实到数据字段与图表。

**(4) 工程可复现性强：导师可在任意机器核查与复画**  
run_id 契约、命令快照、聚合表与定稿图的组合，使得评审者不依赖作者环境即可完成核查；同时也为后续扩展保留了稳定骨架。

### 3.2 贡献：本项目完成的实质贡献

**贡献 1：三场景的规格化构建与可解释化建模**
- RefPlus：可控基线（几何+交通+信号），用于归因对照；  
- UrbMaskPlus：`d_min→b` 连续遮挡强度 + LOS/NLOS 混合 + 反射等效项，捕获同距异质性；  
- TunnelPlus：`tunnel_u` 分段位置 + bell+fade 退化曲线 + 尾延迟注入，捕获分段与 tail。

**贡献 2：拥塞代理量链路与 Cong-only 对照框架**
- 用 `n_cs`/CBR/`p_col`/`cong_delay_ms` 将“交通密度→信道忙→碰撞与排队→tail 变厚”的路径显式化；  
- NoCong 与 Cong 仅差开关，保证差异可归因。

**贡献 3：以 ret=0/1/2 为主杠杆的收益—代价统一评估**
- 不仅展示 PDR 提升，还同时展示 p95/p99 抬升与 late 风险；  
- 使策略选择从“追求更高 PDR”升级为“可靠性与时效性共同约束下的折中”。

**贡献 4：可核查的证据链设计（图—表—字段闭环）**
- Fig05 ⇔ position_heterogeneity ⇔ (`d_min`, `b`, `g_refl`)；  
- Fig06 ⇔ tunnel_segments ⇔ (`tunnel_u`, delay tail)；  
- Fig03/Fig04 ⇔ (`success_phy`, `late`, `p_col`, `cong_delay_ms`)。

### 3.3 可核查性：导师如何快速验证结论可靠

建议按“最短核查路径”阅读与核查：

1) 浏览 Fig01–Fig07（结论闭环）；  
2) 查看 run_commands（协议冻结，NoCong/Cong 差异最小化）；  
3) 核查聚合表字段与口径（tables）；  
4) 若需要下钻，再查看旁证图 F1/F2/F3 与对应表格。

---

## 4 三场景搭建（详细、含方程体系）

本章按照论文的“系统模型/场景设置”写法，从统一坐标与时间离散出发，分别给出 RefPlus/UrbMaskPlus/TunnelPlus 的建模细节，并补充可用于报告表达的方程体系。强调：本项目为系统级评估，目标是在可控复杂度下获得可解释结论，而非替代全栈射线追踪与协议栈仿真。

### 4.0 统一约定：坐标、离散时间与链路中点

- 采用平面坐标：主走廊沿 \(x\) 轴，横向偏移为 \(y\)。  
- 仿真采用离散时间步长 \(\Delta t\)，消息按频率 \(f_m\) 生成。  
- 对 Tx/Rx 形成的链路，定义：  

$$
d=\|\mathbf{p}_{tx}-\mathbf{p}_{rx}\|_2,\qquad 
\mathbf{p}_{mid}=\frac{\mathbf{p}_{tx}+\mathbf{p}_{rx}}{2},\qquad
x_{mid}=\mathbf{p}_{mid,x}.
$$

### 4.1 RefPlus：可控基线（几何 + IDM + 信号灯 + cross/turn）

#### 4.1.1 几何：走廊中心线与多车道

RefPlus 采用 3 km 走廊，S 弯用平滑函数描述（示例表达）：

$$
y_c(x)=A\sin\!\Big(2\pi\,\frac{x-x_0}{x_1-x_0}\Big)\cdot \mathbb{I}_{[x_0,x_1]}(x),
$$

其中 \(A\) 为横向振幅，\([x_0,x_1]\) 为 S 弯区间。车道中心线可写为：

$$
y_{\ell}^{(\pm)}(x)=y_c(x)\ \pm\ \Big(\frac{g}{2}+\big(\ell+\tfrac{1}{2}\big)w\Big),
$$

其中 \(g\) 为中央隔离带宽度，\(w\) 为车道宽度，\(\ell\in\{0,1,2\}\) 为每方向 3 车道索引。

#### 4.1.2 交通：IDM 跟驰模型（产生队列/车团/停走波）

$$
\dot v = a\Big[1-\Big(\frac{v}{v_0}\Big)^{\delta}-\Big(\frac{s^{\ast}(v,\Delta v)}{s}\Big)^2\Big],
\qquad
s^{\ast}(v,\Delta v)=s_0+vT+\frac{v\Delta v}{2\sqrt{ab}},
$$

其中 \(v\) 为本车速度，\(\Delta v\) 为相对速度，\(s\) 为车头间距，\(v_0\) 期望速度，\(T\) 期望时距，\(a,b\) 为最大加速度/舒适减速度，\(s_0\) 为最小间距，\(\delta\) 通常取 4。该模型在信号灯约束下自然产生排队与释放车团，为拥塞代理量 \(n_{cs}\) 提供结构性来源。

#### 4.1.3 信号灯与路口：相位与 offset

设周期 \(C\)，主路绿灯 \(G\)，则主路放行指示函数可写为：

$$
\phi_{main}(t)=\mathbb{I}\big(\ (t+\tau)\bmod C\in[0,G)\ \big),
$$

其中 \(\tau\) 为路口 offset（用于路口相位错开）。在路口区域引入 cross 流与转向概率（右/左/直），以控制交织程度与局部密度峰值。

### 4.2 UrbMaskPlus：城市遮挡/街谷（建筑几何 + 连续遮挡强度 + 反射等效项）

#### 4.2.1 建筑几何：矩形楼块集合

用矩形集合 \(\mathcal{B}=\{B_k\}\) 表示建筑 footprint，每个 \(B_k\) 由 \((x_{min},x_{max},y_{min},y_{max},h)\) 定义。

#### 4.2.2 线段到建筑的最小距离 \(d_{min}\)

$$
d_{min}=\min_{B_k\in\mathcal{B}} \ \mathrm{dist}\big(\overline{\mathbf{p}_{tx}\mathbf{p}_{rx}},\partial B_k\big)
$$

#### 4.2.3 连续遮挡强度 \(b\)

$$
b=\exp\!\left(-\Big(\frac{d_{min}}{T}\Big)^2\right)
$$

#### 4.2.4 LOS/NLOS 混合成功率

$$
p_{succ}(d,b)=(1-b)\,p_{LOS}(d)+b\,p_{NLOS}(d)
$$

#### 4.2.5 反射等效项（可选）

$$
G_{refl}(d_{min},b)=G_{max}\exp(-d_{min}/d_0)\,b
$$

#### 4.2.6 同距异质性统计（固定距离带内按 \(x_{mid}\) 分桶）

$$
\mathcal{S}_j=\{\,\text{links}: d\in[d_1,d_2],\ x_{mid}\in I_j\,\}
$$

### 4.3 TunnelPlus：隧道分段退化（tunnel_u + bell+fade + 尾延迟注入）

#### 4.3.1 隧道归一化位置 \(u\)

$$
u=\mathrm{clip}\Big(\frac{x_{mid}-x_0}{x_1-x_0},0,1\Big)
$$

#### 4.3.2 退化强度形状：bell + fade

$$
\mathrm{bell}(u)=\sin^2(\pi u),\qquad \mathrm{bell}_\gamma(u)=\mathrm{bell}(u)^\gamma
$$

#### 4.3.3 尾延迟注入（示意表达）

$$
D = D_0 + b_{tun}\cdot D_{extra},\qquad D_{extra}\sim \mathrm{Exp}(\lambda)
$$

#### 4.3.4 分段统计（按 \(u\) 分桶）

$$
\mathcal{T}_j=\{\,\text{links}: d\in[d_1,d_2],\ u\in J_j\,\}
$$

### 4.4 证据字段设计（用于“现象—机理—启示”闭环）

- UrbMask：`d_min`、`b`、`g_refl` ⇔ 同距异质性（Fig05）  
- Tunnel：`tunnel_u` ⇔ 分段退化（Fig06）  
- Cong：`n_cs`、CBR、`p_col`、`cong_delay_ms` ⇔ 复合失效拆解（Fig03/04）

---

## 5 机制建模与指标体系

定义 \(S_{phy}\in\{0,1\}\)、\(L\in\{0,1\}\)、\(S=S_{phy}(1-L)\)，并使用：

$$
\mathrm{PDR}\approx \mathrm{PDR}_{phy}\cdot (1-\mathrm{LateRatio}_{phy})
$$

重传与时延叠加的概念表达：

$$
\mathbb{P}(S_{phy}=1)=1-\prod_{k=0}^{ret}(1-p_k),
\qquad
D = D_{base} + D_{prop} + D_{queue} + D_{cong} + D_{retry}
$$

拥塞代理量（示意表达）：

$$
\mathrm{CBR}=\min\big(1,\ (n_{cs}-1)\lambda\tau_{air}\big),
\qquad
p_{col}=1-\exp(-k\,\mathrm{CBR}),
\qquad
D_{cong}\sim \mathrm{Exp}(\beta(\mathrm{CBR}))
$$

---

## 6 可复现性与审计机制（面向评审的严谨表述）

以 `$ROOT` 表示仓库根目录；输出遵循 `runs/<run_id>/` 契约，并包含 `run_commands.txt`。NoCong 与 Cong 的差异被限制为拥塞开关，以保证可归因性。复现建议按 smoke → small → S20 分级执行。

# 结果图解构与讨论（Raw-derived, S20）

## 1. 数据来源与统计流程（raw → cache → figures）

本组结果基于两套严格对照的最终实验：**A_Final_NoCong_S20** 与 **A_Final_Cong_S20**。两套实验的共同配置（由最终命令记录与工程配置文件给出）保持一致：场景集合为 **Ref / UrbMask / Tunnel**，重传策略 **ret=0/1/2**，随机种子 **seed=1–20**，业务统计聚焦于 **0–200 m** 近距离范围。两套实验仅在“是否启用拥塞竞争机制”上存在差异，从而形成严格的消融对照（NoCong vs Cong）。

为了避免“tables 过稀导致曲线不平滑/不可信”的问题，结果并非直接使用 tables，而是采用“两段式”统计管线：
(1) 从 raw 数据中按距离 bin 聚合计数与分位数所需的直方图统计；
(2) 将聚合结果缓存为 `.mat`（便于复现与快速再绘图），随后由 MATLAB 统一绘制 7 张终版图。
在缓存文件命名中可见关键统计分辨率，例如：**distance bin=2 m**、**delay histogram bin=0.25 ms**、机制分析距离带 **band=80–100 m**（用于 UrbMask 空间异质性与 Tunnel 分段机制分析）。

---

## 2. 指标定义与“成功”的层次（避免歧义）

在存在业务时效约束（deadline）的 V2X 场景下，“成功”至少包含两个层次：**物理层成功（PHY success）**与**按时成功（timely success）**。因此，本报告使用三类互补指标：

1. **timely PDR（图中记作 PDR 或 timely_rate）**
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\frac{N*{\mathrm{success}}(d)}{N_{\mathrm{total}}(d)}
$$
其中 (N_{\mathrm{success}}) 表示在 deadline 内到达的成功包数。

2. **PHY success rate（图中记作 phy_rate）**
$$
\mathrm{PDR}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{success_phy}}(d)}{N_{\mathrm{total}}(d)}
$$
其中 (N_{\mathrm{success_phy}}) 表示物理层解码成功的包数（不要求按时）。

3. **late ratio among PHY successes（图中记作 late_ratio_phy）**
$$
\mathrm{late_ratio}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{late}}(d)}{N_{\mathrm{success_phy}}(d)}
$$
其中 (N_{\mathrm{late}}) 表示“PHY 成功但超过 deadline”的包数。

三者满足关键恒等式（Fig03 明确体现）：
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\mathrm{PDR}*{\mathrm{phy}}(d)\cdot\left(1-\mathrm{late_ratio}_{\mathrm{phy}}(d)\right)
$$
该关系将“timely PDR 下降”的原因分解为两部分：**(i) PHY 成功率下降** 与 **(ii) PHY 成功中的超时惩罚上升**，从而支撑“现象—机理—启示”的闭环论证。

---

# Fig01–Fig07 逐图解构（可作为正文 + 图注）

## Fig01：PDR vs distance（dist≤200 m，三场景对照；NoCong 实线，Cong 点线）

![](assets/final_figures_A_preview/Fig01_PDR_Focus.png)

**图意与定位**
Fig01 是整套结果的“主现象图”，用于回答：在退化环境与拥塞竞争背景下，可靠性（timely PDR）随距离如何退化？重传策略（ret）能带来多大收益？

**主要观察 1：NoCong 下的传播主导退化与 ret 的单调增益**
在 NoCong 条件下，三场景曲线均呈随距离单调衰减的典型形态，且 ret 从 0→1→2 造成曲线整体上移，表明重传机制确实提高了按时成功概率。Fig07 的 dist≤200 m 加权汇总给出明确数字锚点：

* Ref：PDR = 0.439（ret0）→ 0.537（ret1）→ 0.582（ret2）
* UrbMask：PDR = 0.462（ret0）→ 0.584（ret1）→ 0.644（ret2）
* Tunnel：PDR = 0.368（ret0）→ 0.489（ret1）→ 0.542（ret2）

该结果表明：在无拥塞竞争时，系统瓶颈主要来自传播成功率（遮挡/隧道退化等），重传能将部分 PHY 失败转化为按时成功，从而形成显著的可靠性增益。

**主要观察 2：Cong 下的全局塌陷与“场景差异二阶化”**
在 Cong 条件下，三场景 PDR 曲线整体被压低到明显更小量级，并且场景之间差异显著缩小。Fig07 的 dist≤200 m 汇总显示：

* Ref：PDR = 0.054（ret0）→ 0.090（ret1）→ 0.117（ret2）
* UrbMask：PDR = 0.057（ret0）→ 0.097（ret1）→ 0.126（ret2）
* Tunnel：PDR = 0.049（ret0）→ 0.082（ret1）→ 0.107（ret2）

这说明：当信道处于忙时竞争状态时，系统主导瓶颈由传播退化转为 **MAC 竞争/碰撞/排队**，传播环境差异（Ref/UrbMask/Tunnel）对最终 timely PDR 的贡献被显著“挤压”为二阶因素。该判断将在 Fig04 的拥塞 proxy 证据中得到进一步支撑。

**报告可用结论句**
NoCong 条件下，ret 提供稳定的可靠性增益并清晰呈现三场景传播差异；Cong 条件下，timely PDR 全局塌陷且场景差异收敛，指示拥塞竞争成为主导瓶颈。

---

## Fig02：Tail delays（p95/p99）vs distance（NoCong/ Cong 分行；ret=0 与 ret=2）

![](assets/final_figures_A_preview/Fig02_TailDelay_p95p99_Ret0Ret2.png)

**图意与定位**
Fig02 是“代价维度”核心图，用于回答：可靠性增强（ret 增大）带来的时延尾部代价有多大？拥塞是否会将尾时延推近 deadline，从而触发 timeliness 惩罚？

**NoCong（上排）：尾时延随 ret 显著上升，但仍远低于 deadline（late≈0）**
上排（NoCong）显示：ret=0 的 p95 处于毫秒级低值，ret=2 的 p95/p99 显著抬升至 10–30 ms 量级，表明重传将成功包的到达时间分布推向更厚尾部。Fig07 的 dist≤200 m p95 锚点如下：

* Ref：p95=2.9 ms（ret0）→ 22.4 ms（ret2）
* UrbMask：p95=3.9 ms（ret0）→ 22.9 ms（ret2）
* Tunnel：p95=3.9 ms（ret0）→ 22.9 ms（ret2）

与此同时，NoCong 的 late=0.0%（Fig07 全为 0.0%），说明在无拥塞时尾时延虽上升，但仍显著低于 deadline，timeliness 惩罚不构成主要失败来源。

**Cong（下排）：尾部被抬升至 70–100 ms 区间，deadline 开始实质参与失败**
下排（Cong）显示：p95 在 70–85 ms 左右，p99 逼近 90–100 ms，尾部整体上移并接近 deadline 量级。Fig07 给出 dist≤200 m 的 p95 锚点：

* Ref：p95=69.6 ms（ret0）→ 79.9 ms（ret2）
* UrbMask：p95=70.1 ms（ret0）→ 79.9 ms（ret2）
* Tunnel：p95=69.1 ms（ret0）→ 79.6 ms（ret2）

因此，Cong 条件下 ret 增大虽提高 PDR（Fig01/Fig07），但也进一步推高尾时延，并引入“PHY 成功但超时”的显性惩罚（late% 上升），该机制将在 Fig03 被定量分解。

**报告可用结论句**
NoCong 场景中可靠性增强主要以“尾时延上升但仍可接受”为代价；Cong 场景中尾时延被推至接近 deadline 的临界区，使 timeliness 惩罚成为可靠性下降的重要组成部分。

---

## Fig03：Cong-only decomposition（PHY success、timely success、late penalty 的分解闭环）

![](assets/final_figures_A_preview/Fig03_Cong_Decomposition_3Curves.png)

**图意与定位**
Fig03 是整套工作“解释力最强”的机理图，用于回答：在 Cong 场景中，timely PDR 为什么如此低？其下降究竟来自 PHY 失败，还是来自超时惩罚？重传 ret 的收益与代价如何在该分解框架中体现？

**读图方式（必须在报告中写清楚）**
每个子图左轴绘制两条曲线：

* 蓝线：(\mathrm{PDR}*{\mathrm{phy}}=N*{\mathrm{success_phy}}/N_{\mathrm{total}})
* 绿线：(\mathrm{PDR}*{\mathrm{timely}}=N*{\mathrm{success}}/N_{\mathrm{total}})
  右轴绘制橙色点线：
* (\mathrm{late_ratio}*{\mathrm{phy}}=N*{\mathrm{late}}/N_{\mathrm{success_phy}})

并由恒等式：
$$
\mathrm{PDR}*{\mathrm{timely}}=\mathrm{PDR}*{\mathrm{phy}}\cdot(1-\mathrm{late_ratio}_{\mathrm{phy}})
$$
将三条曲线联系为统一的因果解释框架。

**主要观察 1：Cong 下“主损失项”首先是 PHY 成功率低（碰撞/竞争导致）**
在 0–200 m 范围内，蓝线（phy_rate）的数值本身已处于低水平，这与 Fig04 中高 p_col 与显著拥塞延迟相一致，表明 Cong 场景的首要损失来自竞争与碰撞造成的 PHY 失败。此时即便 late_ratio 并不极高，timely PDR 也会因 phy_rate 低而被压低（乘法结构）。

**主要观察 2：距离增大后，late_ratio 上升导致 timely_rate 与 phy_rate 进一步拉开**
随着距离增加，橙线（late_ratio_phy）出现抬升趋势，使得绿色曲线（timely）相对蓝色曲线（PHY）出现更明显差距。该现象与 Fig02（p95/p99 接近 deadline）互为印证：拥塞延迟与排队造成的尾部上移，会把一部分 PHY 成功推为超时成功，从而削弱 timely PDR。

**主要观察 3：ret=2 的双刃剑效应可被分解式定量解释**
ret 从 0 到 2 往往会抬升 phy_rate（更多包最终 PHY 成功），因此 timely PDR 会提高；但与此同时 late_ratio 在某些距离段会抬升，使得 timeliness penalty 增加。Fig07 的 late% 数值为该双刃剑提供直接量化：

* Ref：late=5.5%（ret0）→ 8.7%（ret2）
* UrbMask：late=5.4%（ret0）→ 8.4%（ret2）
* Tunnel：late=5.1%（ret0）→ 7.9%（ret2）

因此，Cong 下 ret 的净收益应被理解为“PHY 成功率增益”与“超时惩罚增量”之间的竞争，最终体现在 Fig01/Fig07 的 timely PDR 提升有限而代价显著。

**报告可用结论句**
在 Cong 场景中，timely PDR 的下降可被严格拆解为 PHY success rate 的下降与 late penalty 的上升；重传同时提升 PHY 成功率并增加超时概率，其净效应具有明确的乘法结构与可量化边界。

---

## Fig04：Congestion proxy evidence（均值±标准差带：拥塞强度一致性证据）

![](assets/final_figures_A_preview/Fig04_CongProxy_MeanStdBand.png)

**图意与定位**
Fig04 用于提供“机制证据”：Cong 场景下为何三种传播环境的表现差异缩小？是否确实由于拥塞强度在三场景近似一致，从而使拥塞成为主导瓶颈？

**主要观察：三场景拥塞 proxy 的均值高度一致，标准差带很窄**
三个子图分别给出 avg CBR、avg p_col、avg congestion delay 的均值曲线与跨场景标准差带（±1 std）。在 dist≤200 m 范围内，曲线整体形态一致且方差带较窄，说明 Cong 条件在三场景中构造了相近的“忙时信道强度”。在此背景下，timely PDR 的主要限制来自碰撞/退避/排队，传播差异被“二阶化”，从而解释了 Fig01 中 Cong 曲线在三场景之间的趋同现象。

**报告可用结论句**
Cong 场景下三类拥塞 proxy 在三场景间呈现高一致性（均值接近、方差带窄），从机制上支撑“拥塞主导导致场景差异收敛”的判断。

---

## Fig05：UrbMask spatial heterogeneity（空间异质性与拥塞放大效应）

![](assets/final_figures_A_preview/Fig05_UrbMask_Heterogeneity_LinesAndRatio.png)

**图意与定位**
Fig05 将 UrbMask 的退化特性从“距离域”转向“空间域”，用于回答：城市遮挡是否呈现显著空间异质性？拥塞是否会对这种异质性产生放大效应，从而形成局部不可用区域？

**上图（NoCong）：PDR_band（80–100 m）随 mid_x 出现结构性起伏**
在固定距离带（80–100 m）内，PDR_band 随 mid_x 的变化呈现连续起伏，表明城市遮挡/反射造成的链路质量具有明显位置相关性。ret=2 曲线整体抬升但仍保留空间纹理，说明重传可改善整体可靠性但并不能消除空间非均匀性。

**下图（ratio=Cong/NoCong）：拥塞对空间弱区的退化更强（ratio 更低）**
ratio 在全域显著小于 1，且在某些 mid_x 区段下降更明显。由于 ratio 采用同一位置同一距离带的归一化对照，该结果可被解释为：拥塞不仅降低平均可靠性，还会对特定位置的弱覆盖/弱反射区造成更强惩罚，使局部区域更趋近不可用。这为“退化环境下的风险具有空间集中性”提供了直接证据。

**报告可用结论句**
UrbMask 场景的可靠性退化具有显著空间异质性；拥塞通过降低 Cong/NoCong ratio 的方式对空间弱区施加更强惩罚，提示退化环境风险呈现“位置依赖的局部放大”。

---

## Fig06：Tunnel segmented effect（inside vs outside：分段机制证据与忙时极端化）

![](assets/final_figures_A_preview/Fig06_Tunnel_InsideOutside_Bars_UnifiedY.png)

**图意与定位**
Fig06 用于验证隧道的“分段退化机制”：洞内（inside）是否显著弱于洞外（outside）？拥塞是否会使洞内更接近不可用？同时，ret 的收益与 late 惩罚能否在“inside/outside”对照中被直接量化？

**NoCong（上排）：洞内显著弱于洞外，且 ret 对两者均有改善**
图中柱高为 timely PDR（band 80–100 m），柱上标注 phy 与 late（NoCong late≈0）。典型数值如下（来自图内标注）：

* ret0：inside phy=0.115，outside phy=0.182
* ret1：inside phy=0.214，outside phy=0.333
* ret2：inside phy=0.304，outside phy=0.451
  该结果表明洞内传播条件导致 PHY success rate 明显下降，并直接转化为 timely PDR 的结构性劣化；重传提升 PHY success rate，从而抬升 timely PDR，但 inside/outside 的差距仍保持，体现分段退化的结构性特征。

**Cong（下排）：洞内接近不可用且 late 惩罚出现，ret 提升伴随更高超时风险**
Cong 下排的 inside/outside 柱高更低，且 late>0。典型标注值如下：

* ret0：inside phy=0.015，late=0.059；outside phy=0.034，late=0.030
* ret2：inside phy=0.043，late=0.076；outside phy=0.097，late=0.044
  这表明在忙时竞争下，洞内不仅 PHY 成功率更低，而且 PHY 成功包更容易因排队/退避造成超时，从而削弱 timely PDR。该现象与 Fig02（尾时延逼近 deadline）与 Fig03（late_ratio 上升）形成闭环一致性。

**报告可用结论句**
隧道场景存在稳定的 inside/outside 分段退化：洞内在 NoCong 与 Cong 下均显著弱于洞外；拥塞使洞内进一步逼近不可用，并引入显著 late 惩罚；重传可提升 PHY 成功率但同时增加超时风险。

---

## Fig07：Summary matrices（dist≤200 m 数字锚点与结论收口）

![](assets/final_figures_A_preview/Fig07_Summary_Matrices.png)

**图意与定位**
Fig07 是报告结论的“收口页”，以 dist≤200 m 的加权统计给出每个（场景×ret×拥塞条件）的三元组：**PDR、p95、late%**。该页用于：
(1) 给出可引用的数字锚点；
(2) 统一总结“可靠性—尾时延—timeliness 惩罚”的三维权衡；
(3) 支撑工程启示与策略边界判断。

**NoCong：可靠性增益显著、尾时延代价明确、late≈0**

* Ref：PDR 0.439→0.582；p95 2.9→22.4 ms；late 0.0%
* UrbMask：PDR 0.462→0.644；p95 3.9→22.9 ms；late 0.0%
* Tunnel：PDR 0.368→0.542；p95 3.9→22.9 ms；late 0.0%
  该结果将 NoCong 下“重传增强收益—尾时延代价”的权衡定量钉死。

**Cong：可靠性绝对值低、尾时延接近 deadline、late% 随 ret 上升**

* Ref：PDR 0.054→0.117；p95 69.6→79.9 ms；late 5.5→8.7%
* UrbMask：PDR 0.057→0.126；p95 70.1→79.9 ms；late 5.4→8.4%
* Tunnel：PDR 0.049→0.107；p95 69.1→79.6 ms；late 5.1→7.9%
  该结果表明在忙时竞争下，重传的边际收益显著变小，并且代价从“时延上升”转化为更业务化、更关键的“timeliness 惩罚上升”。

**报告可用结论句**
Fig07 在同一页面内给出可靠性、尾时延与 timeliness 惩罚的联合定量结论：NoCong 条件下重传显著提升可靠性但抬升尾时延；Cong 条件下可靠性绝对值被拥塞主导压低，且重传会提高 late%，提示增强策略需与拥塞控制协同设计。

---

# 跨图综合讨论（顶刊式“现象—机理—启示”闭环）

综合 Fig01/02/07 的主结果与 Fig03/04/05/06 的机理证据，可得到如下闭环结论：

1. **瓶颈迁移（bottleneck shift）**：
   NoCong 条件下瓶颈主要来自传播退化（环境差异清晰可见）；Cong 条件下瓶颈迁移至竞争/碰撞/排队，导致 PDR 全局塌陷并使场景差异二阶化（Fig01 + Fig04）。

2. **增强边界（gain–cost boundary）**：
   重传在 NoCong 下提供显著可靠性增益，但尾时延代价明确且可量化（Fig02 + Fig07）；在 Cong 下重传仍有增益，但会显著增加 late%（timeliness penalty），导致“按时可靠性”的净收益被部分抵消（Fig03 + Fig07）。

3. **退化的结构性（structured degradation）**：
   城市遮挡并非仅导致平均性能下降，而是引入空间异质性；拥塞会放大局部弱区的退化（Fig05）。隧道退化具有明确的 inside/outside 分段结构，且忙时使洞内更趋近不可用（Fig06）。

4. **工程启示（design implication）**：
   在退化环境与忙时竞争共存的真实系统中，仅依赖重传难以同时优化可靠性与及时性；需要将重传与拥塞控制（限速、调度、优先级、资源分配等）联合考虑，使 phy_rate 与 late_ratio 同时受控，从而提升 timely PDR（Fig03 分解框架提供了直接的优化目标）。

---

# 局限性与报告中建议声明（增强严谨性）

为避免审阅者“抓细节”，建议在报告中补充如下声明（可直接复制）：

1. **统计分辨率与平滑**：曲线来自 distance bin 聚合与轻度平滑（仅用于可视化），不改变原始计数与直方图统计；关键数字结论均以 Fig06/Fig07 的加权汇总为准。
2. **分位数估计**：p95/p99 基于 timely success 的延迟直方图聚合估计；当某些距离 bin 的成功样本不足时，对应分位数存在不确定性，因此图中使用筛选/掩蔽策略避免不可靠估计影响结论。
3. **机制 proxy 的定位**：Fig04 所示拥塞 proxy 为模型内可控指标，用于提供“拥塞主导”的证据链，而非对特定标准协议栈的逐细节复现；其价值在于支撑对照解释与策略边界讨论。

## 代码结构概览（快速定位）

工程代码位于 `03_sim_A/py/` 与 `03_sim_A/py/modules/`。整体流水线由 `run_pipeline_A.py` 串联：输入生成 → 仿真 → 聚合 → 出图 → 审计快照。

更细的“逐文件说明（含关键函数/类列表与常见改参入口）”见 Notebook 2。